In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Inicia a construção do objeto principal do Spark (o "ponto de entrada" da aplicação).
# Tudo o que formos fazer com DataFrames passará por esta variável 'spark'.

spark = (
    SparkSession.builder
    # Define o nome da aplicação. É assim que este script aparecerá na Spark Web UI (porta 4040),
    # facilitando o monitoramento de Jobs, Stages e Tasks.
    .appName('01_Motor_Distribuido_Spark')

    # Faz o download automático das bibliotecas Java (JARs) necessárias para o ecossistema.
    # Sem isso, o Spark não sabe se comunicar com S3/Silo nem entende o formato Delta.

    .config(
        'spark.jars.packages',
        'org.apache.hadoop:hadoop-aws:3.3.4,' # Habilita o protocolo s3a://
        'com.amazonaws:aws-java-sdk-bundle:1.12.262,' # SDK da AWS (necessário pelo hadoop-aws)
        'io.delta:delta-spark_2.12:3.2.0' # Motor do Delta Lake para o Spark
    )
    # Endpoint e credenciais do Silo acessível localmente via localhost:9000
    # Redireciona as chamadas que iriam para a nuvem da AWS para o nosso container local.
    .config('spark.hadoop.fs.s3a.endpoint', 'http://localhost:9000')
    .config('spark.hadoop.fs.s3a.access.key', 'admin')
    .config('spark.hadoop.fs.s3a.secret.key', 'password123')

    # Instrui o Spark a usar as credenciais simples que passamos acima, em vez de
    # procurar por credenciais temporárias do IAM ou variáveis de ambiente padrão da AWS.
    .config(
        'spark.hadoop.fs.s3a.aws.credentials.provider',
        'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider'
    )

    # CONFIGURAÇÃO CRÍTICA PARA MINIO/SILO:
    # A AWS usa padrão virtual-hosted (http://meubucket.s3.amazonaws.com).
    # Servidores locais S3-compatíveis usam path-style (http://silo:9000/meubucket).
    # Se 'true', forçamos o padrão path-style para não quebrar a comunicação.
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')

    # Desativa o SSL (HTTPS) pois nossa rede Docker interna não tem certificados.
    # A comunicação ocorrerá via HTTP puro (porta 9000).
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')

    # Mapeia formalmente que qualquer caminho começando com 's3a://' deve ser
    # tratado pela classe Java S3AFileSystem do Hadoop.
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')

    # ==============================================================================
    # CONFIGURAÇÕES DO DELTA LAKE
    # ==============================================================================

    # Injeta os comandos do Delta Lake no motor de SQL nativo do Spark.
    # Isso permite usar comandos como UPDATE, DELETE e MERGE INTO que o Spark puro não possui.
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')

    # Substitui o catálogo padrão de tabelas do Spark pelo catálogo do Delta.
    # Isso é o que permite rastrear o histórico de versões das tabelas (Time Travel).
    .config(
        'spark.sql.catalog.spark_catalog',
        'org.apache.spark.sql.delta.catalog.DeltaCatalog'
    )

    # Comando final que concretiza a construção.
    # "getOrCreate" significa: se já existir uma sessão rodando nesta máquina, pegue ela.
    # Se não existir, crie uma nova com todas as regras acima.
    .getOrCreate()
)

print('SparkSession iniciada com sucesso!')
print('Spark Web UI disponível em: http://localhost:4040')
print(f'Paralelismo padrão da máquina (Cores/Threads): {spark.sparkContext.defaultParallelism}')
print(f'Partições de Shuffle padrão: {spark.conf.get("spark.sql.shuffle.partitions")}')

Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1305)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1291)
	at org.apache.spark.util.Utils$.fetchFile(Utils.scala:432)
	at org.apache.spark.SparkContext.addFile(SparkContext.scala:1878)
	at org.apache.spark.SparkContext.$anonfun$new$17(SparkContext.scala:547)
	at org.apache.spark.SparkContext.$anonfun$new$17$adapted(SparkContext.scala:547)
	at scala.collection.immutable.List.foreach(List.scala:323)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:547)
	at org.apache.spar

Geração dos dados

In [ ]:
import pyspark.sql.functions as F

# ==============================================================================
# 1. GERAÇÃO DE DADOS "SUJOS" (Simulando o Sistema ERP Legado na Origem)
# O objetivo aqui é destruir a qualidade dos dados para treinar o tratamento.
# ==============================================================================

df_base = (
    spark.range(0, 1_000_000)
    .withColumnRenamed('id', 'id_venda')

    # -------------------------------------------------------------------------
    # ERRO INJETADO 1: Inconsistência de Texto (Strings Sujas)
    # -------------------------------------------------------------------------
    # No mundo real, usuários digitam dados de forma livre.
    # O pipeline precisará usar TRIM(), UPPER() ou LOWER() e REPLACE() para padronizar.
    .withColumn(
        'categoria_suja',
        F.element_at(
            F.array(
                F.lit('Eletrônicos'),   # Correto
                F.lit(' livros '),      # Erro: Espaços em branco invisíveis no início e no fim
                F.lit('INFORMÁTICA'),   # Erro: Letras todas maiúsculas
                F.lit('Casa'),          # Correto
                F.lit('Esporte'),       # Correto
                F.lit('eletronicos'),   # Erro: Sem acento e todo minúsculo
                F.lit('CASA')           # Erro: Maiúsculo (o Spark entende 'CASA' e 'Casa' como coisas diferentes!)
            ),
            (F.pmod(F.col('id_venda'), F.lit(7)) + 1).cast('int')
        )
    )

    # -------------------------------------------------------------------------
    # ERRO INJETADO 2: Valores Nulos (Missing Data / NULL)
    # -------------------------------------------------------------------------
    # A cada 50 vendas, forçamos a categoria a vir vazia (None/NULL).

    .withColumn(
        'categoria',
        F.when(F.pmod(F.col('id_venda'), 50) == 0, F.lit(None).cast('string'))
         .otherwise(F.col('categoria_suja'))
    )
    .drop('categoria_suja') # Apaga a coluna auxiliar

    # Geração correta para manter um pouco de dado
    .withColumn(
        'filial_uf',
        F.element_at(
            F.array(F.lit('RS'), F.lit('SC'), F.lit('PR'), F.lit('SP')),
            (F.pmod(F.col('id_venda'), F.lit(4)) + 1).cast('int')
        )
    )
    .withColumn(
        'forma_pagamento',
        F.when(F.pmod(F.col('id_venda'), 3) == 0, 'PIX')
         .when(F.pmod(F.col('id_venda'), 3) == 1, 'Cartão')
         .otherwise('Boleto')
    )
    .withColumn('quantidade', (F.pmod(F.col('id_venda'), 10) + 1).cast('int'))

    # -------------------------------------------------------------------------
    # ERRO INJETADO 3: Anomalias de Negócio (Preços Negativos e Zerados)
    # -------------------------------------------------------------------------
    # - Geramos valores que começam de -50.00. Um preço não pode ser negativo no mundo real.
    # - A cada 33 vendas, o preço vem exatamente 0.00.
    # usar a função filter() para remover essas anomalias.
    .withColumn('preco_base', F.round((F.rand(seed=42) * 500) - 50, 2))
    .withColumn(
        'preco_com_zero',
        F.when(F.pmod(F.col('id_venda'), 33) == 0, F.lit(0.0)).otherwise(F.col('preco_base'))
    )

    # -------------------------------------------------------------------------
    # ERRO INJETADO 4: O Pesadelo da Tipagem e Padrão Regional (Vírgula vs Ponto)
    # -------------------------------------------------------------------------
    # Transformamos o número em um Texto (String). Em metade das linhas


    .withColumn(
        'preco_unitario',
        F.when(F.pmod(F.col('id_venda'), 2) == 0, F.regexp_replace(F.col('preco_com_zero').cast('string'), '\\.', ','))
         .otherwise(F.col('preco_com_zero').cast('string'))
    )
    .drop('preco_base', 'preco_com_zero')

    .withColumn('desconto', F.when(F.col('forma_pagamento') == 'PIX', F.lit(0.05)).otherwise(F.lit(0.0)))
)

# -------------------------------------------------------------------------
# ERRO INJETADO 5: Duplicidade de Registros
# -------------------------------------------------------------------------
# Simulamos uma falha de rede onde o sistema de origem enviou as primeiras
# 10.000 vendas duas vezes. usar .dropDuplicates() depois.
df_gerado_sujo = df_base.unionByName(df_base.limit(10_000))

# ==============================================================================
# 2. GRAVAÇÃO DA ZONA LANDING EM CSV (A Perda de Esquema)
# ==============================================================================
CAMINHO_LANDING_CSV = 's3a://lakehouse/landing/vendas_cruas_csv'

spark.sparkContext.setJobDescription('Gravando Landing Zone (CSV Bruto e Sujo)')
(
    df_gerado_sujo.write
    .format('csv')
    .option('header', 'true')
    # -------------------------------------------------------------------------
    # ERRO INJETADO 6: Escolha do Delimitador
    # -------------------------------------------------------------------------
    # Como colocamos vírgulas nos preços (ex: 150,50), usar a vírgula normal do CSV
    # separaria uma coluna em duas, Por isso usei ponto-e-vírgula.
    .option('sep', ';')
    .mode('overwrite')
    .save(CAMINHO_LANDING_CSV)
)

# Simulando a leitura na área de pouso
df_vendas_landing = spark.read.format('csv').option('header', 'true').option('sep', ';').load(CAMINHO_LANDING_CSV)

print(f"Total de registros na Landing Zone: {df_vendas_landing.count():,} linhas")
df_vendas_landing.show(15)


print("Esquema do CSV (Tudo virou texto):")
df_vendas_landing.printSchema()